# Business Entity Resolution: v14 model (all proven signals of v1 + v6 + v7 + v8 in one model)

**How to run:** **Save Version → Save & Run All (Commit)**. Settings: CPU (no accelerator), Internet On. Input: `er-challenge-data`.
Expected time: about 1.5 h training + 4 to 5 h prediction (under Kaggle's 12 h limit).

**What it learned from earlier versions** (each row was measured on the same 30k held-out businesses as v1):

| Signals used | Val F0.5 |
|---|---|
| v1 signals only (30) | 0.9565 |
| + v6 name commonness (33) | 0.9629 |
| v1 + v7 look-alike checks: house number, legal form, exact keys (61) | 0.9670 |
| v8 sound-alike codes, rare words, rank features (70) | 0.9676 |
| **all together (105)** | **0.9757** |

The different versions catch different mistakes, so one model that sees all their signals beats each of them.
The shortlist also adds v7's exact-key pairs. France has no training answers, so its cut-off is raised by 0.10 to stay on the safe side.
The candidate file keeps each business's **10 best-scored candidates** (plus every match), because the organizers now rank smaller candidate sets higher.

Cells: 1 install, 2 code library, 3 build the training pairs, 4 **train** (prints `BEST val F0.5`), 5 **predict** test and write `output/`, 6 official format check.
No external data or APIs are used. Licenses: LightGBM MIT, XGBoost Apache-2.0, jellyfish MIT.

In [ ]:
!pip install -q rapidfuzz sparse_dot_topn unidecode polars lightgbm xgboost catboost jellyfish

In [ ]:
# =====================================================================
# Business Entity Resolution: full pipeline (clean -> shortlist -> features -> LightGBM -> decide)
# No external data or services are used anywhere. Only the provided files.
# =====================================================================
import os, re, gc, glob, time, json, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import polars as pl
from multiprocessing import Pool
from unidecode import unidecode
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from rapidfuzz.process import cpdist
import lightgbm as lgb

N_CPU = os.cpu_count() or 4

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

# ---------------------------------------------------------------------
# 1. Loading. Files are tab-separated, all columns read as text.
#    Handles files split into *_part_N.tsv (BOM / CRLF / repeated header).
# ---------------------------------------------------------------------
def find_file(root, name):
    hits = sorted(glob.glob(f"{root}/**/{name}.tsv", recursive=True))
    if hits:
        return [hits[0]]
    parts = glob.glob(f"{root}/**/{name}_part_*.tsv", recursive=True)
    parts.sort(key=lambda p: int(re.findall(r"_part_(\d+)", p)[-1]))
    if not parts:
        raise FileNotFoundError(f"{name}.tsv not found under {root}")
    return parts

def load(root, name):
    frames, header = [], None
    for p in find_file(root, name):
        txt = open(p, encoding="utf-8-sig").read().replace("\r\n", "\n").replace("\r", "")
        if txt.startswith(("entity_id\t", "source1_entity_id\t")):
            header, txt = txt.split("\n", 1)
        frames.append(txt)
    body = "".join(f if f.endswith("\n") else f + "\n" for f in frames)
    df = pl.read_csv((header + "\n" + body).encode(), separator="\t", quote_char=None, infer_schema_length=0)
    return df.fill_null("").to_pandas()

# ---------------------------------------------------------------------
# 2. Cleaning (normalisation). Hand-written rules only.
# ---------------------------------------------------------------------
ABBR = {"rd":"road","st":"street","ave":"avenue","av":"avenue","blvd":"boulevard","bd":"boulevard","hwy":"highway",
        "dr":"drive","ln":"lane","ct":"court","pl":"place","pkwy":"parkway","cir":"circle","ste":"suite","fl":"floor",
        "apt":"apartment","no":"number","nr":"near","opp":"opposite","bldg":"building","ind":"industrial",
        "dist":"district","r":"rue","ch":"chemin","n":"north","s":"south","e":"east","w":"west","mt":"mount",
        "ft":"fort","hts":"heights","sq":"square","imp":"impasse","all":"allee"}
NULLS = {"null", "na", "none", "nan"}
# legal words, written in "skeleton" form (see skel_tok): covers Limited/Ltd/limittedd, Private/Pvt/praaivett, LLC, SARL ...
LEGAL_SK = {"lmtd","ltd","prwt","pwt","llc","llp","inc","ncrprtd","crp","crprtn","c","cmpny","plc","srl","ss","sa",
            "sc","erl","snc","gmbh","t","m","s","d","dba","lp","pc","pllc","pa","ms","and"}

def base(s):
    s = unidecode(str(s)).lower()                                   # also turns Hindi/Telugu/... script into Latin letters
    s = re.sub(r"<?null>?|n/a", " ", s)
    s = re.sub(r"https?://|www\.|\.(com|net|org|in|co|io|biz|fr|us)\b", " ", s)
    s = s.replace("&", " and ").replace("+", " and ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def skel_tok(t):
    """Spelling-robust 'skeleton' of a word: drop vowels, merge ph/f, bh/b ..., collapse double letters.
    'limited' and the transliterated 'limittedd' both become 'lmtd'."""
    if not t or t.isdigit():
        return t
    t = t.replace("0", "o").replace("1", "l") if re.search(r"[a-z]", t) else t   # leetspeak: C1ean -> Clean
    t = re.sub(r"ph", "f", t)
    t = re.sub(r"([bkgtdcs])h", r"\1", t)
    t = t.replace("v", "w").replace("q", "k").replace("z", "s").replace("x", "ks")
    t = t[0] + re.sub(r"[aeiouy]", "", t[1:])
    return re.sub(r"(.)\1+", r"\1", t)

US = {"alabama":"al","alaska":"ak","arizona":"az","arkansas":"ar","california":"ca","colorado":"co","connecticut":"ct",
"delaware":"de","florida":"fl","georgia":"ga","hawaii":"hi","idaho":"id","illinois":"il","indiana":"in","iowa":"ia",
"kansas":"ks","kentucky":"ky","louisiana":"la","maine":"me","maryland":"md","massachusetts":"ma","michigan":"mi",
"minnesota":"mn","mississippi":"ms","missouri":"mo","montana":"mt","nebraska":"ne","nevada":"nv","new hampshire":"nh",
"new jersey":"nj","new mexico":"nm","new york":"ny","north carolina":"nc","north dakota":"nd","ohio":"oh","oklahoma":"ok",
"oregon":"or","pennsylvania":"pa","rhode island":"ri","south carolina":"sc","south dakota":"sd","tennessee":"tn",
"texas":"tx","utah":"ut","vermont":"vt","virginia":"va","washington":"wa","west virginia":"wv","wisconsin":"wi",
"wyoming":"wy","district of columbia":"dc","puerto rico":"pr"}
IN = {"andhra pradesh":"ap","arunachal pradesh":"ar","assam":"as","bihar":"br","chhattisgarh":"cg","goa":"ga",
"gujarat":"gj","haryana":"hr","himachal pradesh":"hp","jharkhand":"jh","karnataka":"ka","kerala":"kl",
"madhya pradesh":"mp","maharashtra":"mh","manipur":"mn","meghalaya":"ml","mizoram":"mz","nagaland":"nl","odisha":"od",
"orissa":"od","punjab":"pb","rajasthan":"rj","sikkim":"sk","tamil nadu":"tn","telangana":"ts","tripura":"tr",
"uttar pradesh":"up","uttarakhand":"uk","uttaranchal":"uk","west bengal":"wb","delhi":"dl","new delhi":"dl",
"jammu and kashmir":"jk","ladakh":"la","puducherry":"py","pondicherry":"py","chandigarh":"ch",
"andaman and nicobar islands":"an","lakshadweep":"ld"}
MERGE_ST = {"ts": "ap"}            # Telangana was split from Andhra Pradesh; the data mixes them
_STATE = {}
for _d in (US, IN):
    for _full, _code in _d.items():
        _STATE[tuple(skel_tok(t) for t in _full.split())] = _code
_CODES = set(US.values()) | set(IN.values())

def states_of(raw_addr):
    """All state/province codes mentioned in an address ('' when none). Country-agnostic: unknown countries get ''."""
    found = set()
    for seg in unidecode(str(raw_addr)).lower().split(","):
        toks = re.sub(r"[^a-z ]", " ", seg).split()
        if not toks:
            continue
        if len(toks) == 1 and toks[0] in _CODES:
            found.add(toks[0]); continue
        sk = [skel_tok(t) for t in toks]
        for n in (4, 3, 2, 1):
            for i in range(len(sk) - n + 1):
                key = tuple(sk[i:i + n])
                if key in _STATE and (n > 1 or len(key[0]) >= 3):
                    found.add(_STATE[key])
    return "|".join(sorted({MERGE_ST.get(x, x) for x in found}))

def _clean_rows(rows):
    out = []
    for name, addr in rows:
        nb = base(name)
        sk = [skel_tok(t) for t in nb.split()]
        core = [t for t in sk if t not in LEGAL_SK] or sk
        an = " ".join(ABBR.get(t, t) for t in base(addr).split() if t not in NULLS)
        out.append((nb, " ".join(core), "".join(core), an,
                    " ".join(skel_tok(t) for t in an.split()), states_of(addr)))
    return out

def prepare(df):
    """Adds the cleaned columns. Runs on all CPU cores."""
    rows = list(zip(df.business_name.tolist(), df.business_address.tolist()))
    step = 50_000
    with Pool(N_CPU) as pool:
        res = pool.map(_clean_rows, [rows[i:i + step] for i in range(0, len(rows), step)])
    res = [r for chunk in res for r in chunk]
    cols = ["name_b", "name_sk", "name_skc", "addr_n", "addr_sk", "states"]
    for j, c in enumerate(cols):
        df[c] = [r[j] for r in res]
    df.drop(columns=["business_address"], inplace=True)
    df["nonascii"] = df.business_name.str.contains(r"[^\x00-\x7f]", regex=True).astype(np.int8)
    df.drop(columns=["business_name"], inplace=True)
    return df.reset_index(drop=True)

# ---------------------------------------------------------------------
# 3. Shortlist ("blocking"). For each Source 1 record, look only inside the same country and
#    the same state, and keep the top-k most similar names and the top-k most similar addresses.
#    Records whose state could not be read form a separate pool per country.
# ---------------------------------------------------------------------
def _vec_name():
    return TfidfVectorizer(analyzer="char", ngram_range=(3, 3), dtype=np.float32, sublinear_tf=True)
def _vec_addr():
    return TfidfVectorizer(analyzer="word", token_pattern=r"\S+", ngram_range=(1, 2), dtype=np.float32, sublinear_tf=True)

class Index:
    """TF-IDF indexes over one pool of candidate records (one country of Source 2 or Source 3)."""
    def __init__(self, B, k_state=15, k_unk=10):
        self.B, self.k_state, self.k_unk = B, k_state, k_unk
        self.cache = {}
        st = B.states.str.split("|")
        self.members = {}
        for pos, lst in zip(range(len(B)), st):
            for s in lst:
                if s:
                    self.members.setdefault(s, []).append(pos)
        self.unknown = np.flatnonzero(B.states.to_numpy() == "")

    def _build(self, key, pos):
        if key in self.cache:
            return self.cache[key]
        pos = np.asarray(pos, dtype=np.int64)
        entry = {"pos": pos}
        for col, mk in (("name_skc", _vec_name), ("addr_sk", _vec_addr)):
            try:
                v = mk(); M = v.fit_transform(self.B[col].to_numpy()[pos])
                entry[col] = (v, M.T.tocsr())
            except ValueError:            # empty vocabulary, e.g. every address blank
                entry[col] = None
        self.cache[key] = entry
        return entry

    def query(self, A, thr=0.1, chunk=40_000):
        """Returns DataFrame(a, b, sim_name, sim_addr); a = row in A, b = row in self.B."""
        out = []
        ast = A.states.str.split("|").tolist()
        groups = {}
        for i, lst in enumerate(ast):
            for s in lst:
                groups.setdefault(s, []).append(i)
        jobs = []
        for s, rows in groups.items():
            if s == "":
                jobs.append(("__all__", np.arange(len(self.B)), rows, self.k_state))
            elif s in self.members:
                jobs.append((s, self.members[s], rows, self.k_state))
        known = [i for i, lst in enumerate(ast) if lst != [""]]
        if len(self.unknown) and known:
            jobs.append(("__unknown__", self.unknown, known, self.k_unk))
        for key, pos, rows, k in jobs:
            e = self._build(key, pos)
            rows = np.asarray(rows)
            for col, tag in (("name_skc", 0), ("addr_sk", 1)):
                if e[col] is None:
                    continue
                v, BT = e[col]
                texts = A[col].to_numpy()[rows]
                for i in range(0, len(rows), chunk):
                    C = sp_matmul_topn(v.transform(texts[i:i + chunk]), BT, top_n=k, threshold=thr,
                                       n_threads=N_CPU).tocoo()
                    out.append((rows[i + C.row], e["pos"][C.col], C.data, tag))
        if not out:
            return pd.DataFrame({"a": [], "b": [], "sim_name": [], "sim_addr": []})
        a = np.concatenate([o[0] for o in out]).astype(np.int64)
        b = np.concatenate([o[1] for o in out]).astype(np.int64)
        sim = np.concatenate([o[2] for o in out]).astype(np.float32)
        tag = np.concatenate([np.full(len(o[0]), o[3], np.int8) for o in out])
        df = pd.DataFrame({"key": a * len(self.B) + b,
                           "sim_name": np.where(tag == 0, sim, np.nan),
                           "sim_addr": np.where(tag == 1, sim, np.nan)})
        df = df.groupby("key", sort=False).max().reset_index()
        df["a"] = (df.key // len(self.B)).astype(np.int32)
        df["b"] = (df.key % len(self.B)).astype(np.int32)
        return df.drop(columns="key")

# ---------------------------------------------------------------------
# 4. Features: how similar is each (Source 1, candidate) pair?
# ---------------------------------------------------------------------
FULL = [("tset", fuzz.token_set_ratio), ("tsort", fuzz.token_sort_ratio), ("ratio", fuzz.ratio),
        ("partial", fuzz.partial_ratio), ("jw", JaroWinkler.normalized_similarity)]

def _nums(s):
    return set(re.findall(r"\d+", s))

def features(P, A, B, is_s3):
    a, b = P.a.to_numpy(), P.b.to_numpy()
    g = lambda D, col, idx: D[col].to_numpy()[idx]
    f = pd.DataFrame({"sim_name": P.sim_name.to_numpy(), "sim_addr": P.sim_addr.to_numpy()})
    for col, pre, scorers in (("name_b", "nb", FULL), ("name_sk", "nsk", FULL),
                              ("name_skc", "nskc", [("ratio", fuzz.ratio), ("partial", fuzz.partial_ratio)]),
                              ("addr_sk", "ask", [("tset", fuzz.token_set_ratio), ("ratio", fuzz.ratio),
                                                  ("partial", fuzz.partial_ratio)])):
        x, y = g(A, col, a).tolist(), g(B, col, b).tolist()
        for nm, sc in scorers:
            f[f"{pre}_{nm}"] = cpdist(x, y, scorer=sc, workers=N_CPU).astype(np.float32)
    xa, ya = g(A, "addr_n", a), g(B, "addr_n", b)
    nl = [_nums(s) for s in xa]; nr = [_nums(s) for s in ya]
    f["num_jacc"] = np.array([len(p & q) / len(p | q) if (p | q) else -1 for p, q in zip(nl, nr)], np.float32)
    f["num_contain"] = np.array([len(p & q) / len(q) if q else -1 for p, q in zip(nl, nr)], np.float32)
    sa, sb = g(A, "states", a), g(B, "states", b)
    f["state_rel"] = np.array([-1 if (not p or not q) else (1 if set(p.split("|")) & set(q.split("|")) else 0)
                               for p, q in zip(sa, sb)], np.int8)
    f["name_exact"] = (g(A, "name_skc", a) == g(B, "name_skc", b)).astype(np.int8)
    f["cand_addr_len"] = np.fromiter((len(s) for s in ya), np.int32, len(ya))
    f["s1_addr_len"] = np.fromiter((len(s) for s in xa), np.int32, len(xa))
    f["cand_name_len"] = np.fromiter((len(s) for s in g(B, "name_b", b)), np.int32, len(b))
    f["s1_name_len"] = np.fromiter((len(s) for s in g(A, "name_b", a)), np.int32, len(a))
    f["cand_nonascii"] = g(B, "nonascii", b)
    f["is_s3"] = np.int8(is_s3)
    # context: how this candidate compares with the other candidates of the same Source 1 record
    v = f.nsk_tset.to_numpy() + f.ask_tset.to_numpy()
    grp = pd.DataFrame({"a": a, "v": v}).groupby("a")["v"]
    f["rank_s1"] = grp.rank(ascending=False, method="min").to_numpy(np.float32)
    f["gap_s1"] = (grp.transform("max").to_numpy() - v).astype(np.float32)
    f["n_cand_s1"] = grp.transform("size").to_numpy(np.int32)
    return f

# ---------------------------------------------------------------------
# 5. Scoring (same formula as the leaderboard) and the final decision rule
# ---------------------------------------------------------------------
def macro_f05(pred, truth, ids):
    sc = []
    for s in ids:
        p, t = pred.get(s, set()), truth.get(s, set())
        if not t:
            sc.append(1.0 if not p else 0.0); continue
        tp = len(p & t)
        if tp == 0:
            sc.append(0.0); continue
        P, R = tp / len(p), tp / len(t)
        sc.append(1.25 * P * R / (0.25 * P + R))
    return float(np.mean(sc))

def decide(pairs, thr):
    """pairs: DataFrame(s1_id, cand_id, p). Keep p >= thr; each S2/S3 record goes to at most one S1 (its best)."""
    d = pairs[pairs.p >= thr].sort_values("p", ascending=False).drop_duplicates("cand_id")
    return d.groupby("s1_id")["cand_id"].apply(set).to_dict()

def write_tsv(path, s1_ids, mapping, col):
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(f"source1_entity_id\t{col}\n")
        for s in s1_ids:
            fh.write(s + "\t" + ",".join(sorted(mapping.get(s, ()))) + "\n")

# =====================================================================
# v8 additions: richer pair features + an ensemble of gradient boosting models.
# Everything here works on top of the v1 pipeline (erpipe) and uses no external data.
# Libraries: jellyfish (MIT), xgboost (Apache-2.0), catboost (Apache-2.0), lightgbm (MIT).
# =====================================================================
import jellyfish
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer
import xgboost as xgb
from catboost import CatBoostClassifier

# ---------------------------------------------------------------------
# 1. Extra cleaned columns per record
#    name_w  : the name's words without legal words (Pvt, Ltd, LLC, SARL ...)
#    name_ph : the same words as sound codes (metaphone), so "Sharma" and "Sharmaa" give the same code
# ---------------------------------------------------------------------
def _extra_rows(names):
    out = []
    for nb in names:
        toks = nb.split()
        core = [t for t in toks if skel_tok(t) not in LEGAL_SK] or toks
        ph = [jellyfish.metaphone(t) if not t.isdigit() else t for t in core]
        out.append((" ".join(core), " ".join(p for p in ph if p)))
    return out

def prepare_extra(df):
    names = df.name_b.tolist(); step = 100_000
    with Pool(N_CPU) as pool:
        res = pool.map(_extra_rows, [names[i:i + step] for i in range(0, len(names), step)])
    res = [r for chunk in res for r in chunk]
    df["name_w"] = [r[0] for r in res]
    df["name_ph"] = [r[1] for r in res]
    return df

# ---------------------------------------------------------------------
# 2. Word-rarity ("IDF") weights, learned from the candidate pool itself.
#    A word shared by 50,000 businesses ("store", "road") says little; a rare word ("zanskar") says a lot.
# ---------------------------------------------------------------------
class Rarity:
    """Word vocabularies + rarity weights for one candidate pool (one source, one country)."""
    def __init__(self, B):
        self.parts = {}
        for col in ("name_w", "addr_n", "name_ph"):
            v = CountVectorizer(token_pattern=r"\S+", binary=True, dtype=np.float32)
            try:
                X = v.fit_transform(B[col].to_numpy())
            except ValueError:            # empty vocabulary
                self.parts[col] = None; continue
            df = np.asarray(X.sum(axis=0)).ravel()
            idf = (np.log((len(B) + 1) / (df + 1)) + 1).astype(np.float32)
            self.parts[col] = (v, idf, X.tocsr(), float(np.log(len(B) + 1) + 1))
        # full char 3-gram similarity of the name skeleton for EVERY pair (the shortlist only kept top-15)
        self.cv = TfidfVectorizer(analyzer="char", ngram_range=(3, 3), dtype=np.float32, sublinear_tf=True)
        try:
            self.CB = self.cv.fit_transform(B.name_skc.to_numpy()).tocsr()
        except ValueError:
            self.cv = None

def _rowdot(XA, XB):
    return np.asarray(XA.multiply(XB).sum(axis=1)).ravel().astype(np.float32)

def _rowmax(X):
    if X.nnz == 0:
        return np.zeros(X.shape[0], np.float32)
    return np.asarray(X.max(axis=1).todense()).ravel().astype(np.float32)

def rarity_features(R, a, b, A, f, chunk=3_000_000):
    """Adds rarity-weighted similarities for (A row a, pool row b) pairs into DataFrame f."""
    for col, pre in (("name_w", "rn"), ("addr_n", "ra"), ("name_ph", "rp")):
        names = [f"{pre}_{x}" for x in ("wjacc", "cont_s1", "cont_c", "cos", "miss_s1", "miss_c", "share_max")]
        if R.parts[col] is None:
            for n in names: f[n] = np.float32(np.nan)
            continue
        v, idf, XB_all, max_idf = R.parts[col]
        texts = A[col].to_numpy()
        XA_all = v.transform(texts).tocsr()
        # words of the S1 record that never occur in the pool: maximally rare, and surely not shared
        n_unseen = np.array([len(set(t.split())) for t in texts], np.float32) - np.asarray(XA_all.sum(axis=1)).ravel()
        W = sp.diags(idf); W2 = sp.diags(idf * idf)
        XAw, XBw = (XA_all @ W).tocsr(), (XB_all @ W).tocsr()
        sA = np.asarray(XAw.sum(axis=1)).ravel() + n_unseen * max_idf
        sB = np.asarray(XBw.sum(axis=1)).ravel()
        nA = np.sqrt(np.asarray((XA_all @ W2).sum(axis=1)).ravel() + n_unseen * max_idf ** 2)
        nB = np.sqrt(np.asarray((XB_all @ W2).sum(axis=1)).ravel())
        res = {n: np.empty(len(a), np.float32) for n in names}
        for i in range(0, len(a), chunk):
            ai, bi = a[i:i + chunk], b[i:i + chunk]
            PA, PB = XAw[ai], XB_all[bi]
            S = PA.multiply(PB).tocsr()                               # shared words, with their weights
            inter = np.asarray(S.sum(axis=1)).ravel()
            inter2 = np.asarray(S.multiply(S).sum(axis=1)).ravel()
            sa, sb = sA[ai], sB[bi]
            with np.errstate(divide="ignore", invalid="ignore"):
                res[names[0]][i:i + chunk] = np.where(sa + sb - inter > 0, inter / (sa + sb - inter), np.nan)
                res[names[1]][i:i + chunk] = np.where(sa > 0, inter / sa, np.nan)
                res[names[2]][i:i + chunk] = np.where(sb > 0, inter / sb, np.nan)
                res[names[3]][i:i + chunk] = np.where(nA[ai] * nB[bi] > 0, inter2 / (nA[ai] * nB[bi]), np.nan)
            missA = _rowmax((PA - PA.multiply(PB > 0)).tocsr())        # rarest S1 word the candidate lacks
            res[names[4]][i:i + chunk] = np.maximum(missA, np.where(n_unseen[ai] > 0, max_idf, 0))
            PBw = XBw[bi]
            res[names[5]][i:i + chunk] = _rowmax((PBw - PBw.multiply(PA > 0)).tocsr())   # rarest candidate word S1 lacks
            res[names[6]][i:i + chunk] = _rowmax(S)                   # rarest word they share
        for n in names: f[n] = res[n]
    if R.cv is not None:
        CA = R.cv.transform(A.name_skc.to_numpy()).tocsr()
        out = np.empty(len(a), np.float32)
        for i in range(0, len(a), chunk):
            out[i:i + chunk] = _rowdot(CA[a[i:i + chunk]], R.CB[b[i:i + chunk]])
        f["nch_cos"] = out
    else:
        f["nch_cos"] = np.float32(np.nan)

def phonetic_features(a, b, A, B, f):
    x, y = A.name_ph.to_numpy()[a].tolist(), B.name_ph.to_numpy()[b].tolist()
    f["ph_tset"] = cpdist(x, y, scorer=fuzz.token_set_ratio, workers=N_CPU).astype(np.float32)
    f["ph_ratio"] = cpdist(x, y, scorer=fuzz.ratio, workers=N_CPU).astype(np.float32)
    f["ph_first_eq"] = np.array([(p.split(" ", 1)[0] == q.split(" ", 1)[0]) if p and q else -1
                                 for p, q in zip(x, y)], np.int8)
    xw, yw = A.name_w.to_numpy()[a].tolist(), B.name_w.to_numpy()[b].tolist()
    f["nw_tset"] = cpdist(xw, yw, scorer=fuzz.token_set_ratio, workers=N_CPU).astype(np.float32)

# ---------------------------------------------------------------------
# 3. Rank features: how does this candidate compare with the business's other candidates (same source)?
# ---------------------------------------------------------------------
RANK_ON = ["rn_cos", "ra_cos", "nch_cos", "ph_tset", "combo"]

def rank_features(a, f):
    f["combo"] = (f.rn_cos.fillna(0) + f.ra_cos.fillna(0) + f.nch_cos.fillna(0) + f.ph_tset / 100).astype(np.float32)
    g = pd.DataFrame({"a": a})
    for k in RANK_ON:
        g["v"] = f[k].fillna(-1).to_numpy()
        grp = g.groupby("a")["v"]
        mx = grp.transform("max").to_numpy()
        f[f"rk_{k}"] = grp.rank(ascending=False, method="min").to_numpy(np.float32)
        f[f"gap_{k}"] = (mx - g.v.to_numpy()).astype(np.float32)
    v = g.v.to_numpy()                      # g.v is "combo" now (last in RANK_ON)
    # runner-up value per business, found by sorting once
    order = np.lexsort((-v, a)); a_s, v_s = a[order], v[order]
    first = np.r_[True, a_s[1:] != a_s[:-1]]
    pos = np.arange(len(a_s)); start = np.maximum.accumulate(np.where(first, pos, 0))
    has2 = (start + 1 < len(a_s)) & (a_s[np.minimum(start + 1, len(a_s) - 1)] == a_s)
    sec_s = np.where(has2, v_s[np.minimum(start + 1, len(a_s) - 1)], -1)
    second = np.empty_like(v); second[order] = sec_s
    # top candidate: lead over the runner-up; others: distance behind the leader (negative)
    f["combo_lead"] = np.where(v >= mx, v - second, v - mx).astype(np.float32)
    f["combo_z"] = ((g.v - grp.transform("mean")) / (grp.transform("std").fillna(0) + 1e-3)).to_numpy(np.float32)
    f["n_close"] = g.assign(c=(mx - v) < 0.15).groupby("a")["c"].transform("sum").to_numpy(np.int16)

def features_v8(P, A, B, is_s3, R):
    """v1 features + v8 features for the shortlist pairs P of A (S1 batch) against pool B."""
    f = features(P, A, B, is_s3)
    a, b = P.a.to_numpy(), P.b.to_numpy()
    rarity_features(R, a, b, A, f)
    phonetic_features(a, b, A, B, f)
    rank_features(a, f)
    return f

# ---------------------------------------------------------------------
# 4. The ensemble: several boosting models, each gives a probability; the blend is their average.
# ---------------------------------------------------------------------
MODEL_SPECS = [
    ("lgb_a", "lgb", dict(learning_rate=0.05, num_leaves=255, min_child_samples=100, subsample=0.8, subsample_freq=1,
                          colsample_bytree=0.8, random_state=0)),
    ("lgb_b", "lgb", dict(learning_rate=0.05, num_leaves=127, min_child_samples=50, subsample=0.7, subsample_freq=1,
                          colsample_bytree=0.5, reg_lambda=5.0, random_state=1)),
    ("xgb", "xgb", dict(eta=0.08, max_depth=9, min_child_weight=5, subsample=0.8, colsample_bytree=0.7,
                        tree_method="hist", max_bin=256, eval_metric="logloss", objective="binary:logistic", seed=2)),
    ("cat", "cat", dict(learning_rate=0.12, depth=8, l2_leaf_reg=3, random_seed=3, border_count=128)),
]

def fit_model(kind, params, Xtr, ytr, Xva, yva, max_rounds, n_rounds=None):
    """Trains one model. With n_rounds=None it stops when the held-out loss stops improving."""
    es = n_rounds is None
    rounds = max_rounds if es else n_rounds
    if kind == "lgb":
        m = lgb.LGBMClassifier(n_estimators=rounds, n_jobs=N_CPU, verbose=-1, **params)
        m.fit(Xtr, ytr, **({"eval_set": [(Xva, yva)], "callbacks": [lgb.early_stopping(100, verbose=False)]} if es else {}))
        return m, (m.best_iteration_ if es else rounds)
    if kind == "xgb":
        dtr = xgb.QuantileDMatrix(Xtr, ytr, max_bin=params.get("max_bin", 256), nthread=N_CPU)
        ev = [(xgb.DMatrix(Xva, yva, nthread=N_CPU), "val")] if es else []
        m = xgb.train({**params, "nthread": N_CPU}, dtr, rounds, evals=ev,
                      early_stopping_rounds=100 if es else None, verbose_eval=False)
        del dtr; gc.collect()
        return m, (m.best_iteration + 1 if es else rounds)
    if kind == "cat":
        m = CatBoostClassifier(iterations=rounds, thread_count=N_CPU, verbose=False, allow_writing_files=False,
                               **({"od_type": "Iter", "od_wait": 100, "use_best_model": True} if es else {}), **params)
        m.fit(Xtr, ytr, eval_set=(Xva, yva) if es else None)
        return m, (m.get_best_iteration() + 1 if es else rounds)

def predict_model(kind, m, X):
    if kind == "lgb":
        return m.predict(X, num_threads=N_CPU).astype(np.float32) if isinstance(m, lgb.Booster) \
            else m.predict_proba(X)[:, 1].astype(np.float32)
    if kind == "xgb":
        return m.predict(xgb.DMatrix(X, nthread=N_CPU)).astype(np.float32)
    return m.predict_proba(X)[:, 1].astype(np.float32)

def save_model(kind, m, path):
    if kind == "lgb": m.booster_.save_model(path)
    elif kind == "xgb": m.save_model(path)
    else: m.save_model(path)

def load_model(kind, path):
    if kind == "lgb": return lgb.Booster(model_file=path)
    if kind == "xgb":
        m = xgb.Booster(); m.load_model(path); m.set_param({"nthread": N_CPU}); return m
    m = CatBoostClassifier(); m.load_model(path); return m

MODEL_EXT = {"lgb": "txt", "xgb": "json", "cat": "cbm"}

# =====================================================================
# Version 7: exact "keys" (rule layer)
# A key is a short piece of text built from a record, e.g. "house number + street word + one word of the name".
# Two records that share a rare key are very likely the same business.
# We use keys in three ways:
#   1. extra shortlist: pairs sharing a key are added even if the TF-IDF shortlist missed them
#   2. extra features for LightGBM: which keys matched, and how common each key is
#   3. rules: key types that were (almost) always right on training data accept a pair directly
# Needs the functions from the v1 pipeline cell (prepare, skel_tok, log, N_CPU ...).
# =====================================================================
from hashlib import blake2b
import ctypes

def free_mem():
    """Free memory and hand it back to the system (so helper processes don't copy it)."""
    gc.collect()
    try: ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception: pass

KEY_TYPES = ["name", "nameset", "name_num", "addr", "addr_tok", "tok2_num"]
KCAP = 30                    # keys shared by more than this many records on one side are too common to use

def _ktok(t):
    """Extra-strong spelling key for one word: skeleton, and c/k, w/v, j/z sound the same.
    ('healthcare' and the transliterated Hindi 'helthker' both become 'hltkr')."""
    return skel_tok(t).replace("c", "k").replace("j", "z")

def _h(s):
    return int.from_bytes(blake2b(s.encode(), digest_size=8).digest(), "little", signed=True)

def _keys_rows(args):
    rows, start = args
    ri, kt, kh = [], [], []
    for off, (country, name_sk, addr_n) in enumerate(rows):
        r = start + off
        toks = [_ktok(t) for t in name_sk.split() if t]
        toks = [t for t in toks if t]
        ks = []
        cat = "".join(toks)
        if len(cat) >= 4:
            ks.append((0, cat))
            ks.append((1, "".join(sorted(set(toks)))))
        at = addr_n.split()
        nums = []
        for i, t in enumerate(at):                          # every number in the address + the next word with letters
            d = re.sub(r"\D", "", t)
            if not d or len(nums) >= 3:
                continue
            d = d.lstrip("0") or "0"
            nxt = next((_ktok(x) for x in at[i + 1:i + 4] if re.search(r"[a-z]", x)), "")
            nums.append((d, nxt))
        big = [t for t in toks if len(t) >= 3 and not t.isdigit()][:4]
        for d, nxt in dict.fromkeys(nums):
            if len(cat) >= 4:
                ks.append((2, cat + "|" + d))
            if nxt:
                ks.append((3, d + "|" + nxt))
                for t in big:
                    ks.append((4, d + "|" + nxt + "|" + t))
            if len(big) >= 2:
                ks.append((5, big[0] + "|" + big[1] + "|" + d))
        for t, s in dict.fromkeys(ks):
            ri.append(r); kt.append(t); kh.append(_h(f"{country}|{t}|{s}"))
    return np.array(ri, np.int32), np.array(kt, np.int8), np.array(kh, np.int64)

def make_keys(D):
    """All keys of every record in a prepared table. Returns polars DataFrame(row, kt, kh, cnt)
    where cnt = how many records of this table share the key."""
    rows = list(zip(D.country.tolist(), D.name_sk.tolist(), D.addr_n.tolist()))
    step = 50_000
    with Pool(N_CPU) as pool:
        res = pool.map(_keys_rows, [(rows[i:i + step], i) for i in range(0, len(rows), step)])
    K = pl.DataFrame({"row": np.concatenate([r[0] for r in res]), "kt": np.concatenate([r[1] for r in res]),
                      "kh": np.concatenate([r[2] for r in res])})
    K = K.with_columns(pl.len().over("kh").cast(pl.Int32).alias("cnt"))
    return K

def key_pairs(KA, KB, a_rows=None):
    """Pairs (a, b) that share at least one key, with per key type: hit, how many S1 / candidates share it.
    KA: keys of Source 1 (row = row in the S1 table); KB: keys of the candidate table.
    a_rows: optional array of S1 rows to keep (a batch); output 'a' is then the position inside a_rows."""
    ka = KA.filter(pl.col("cnt") <= KCAP)
    if a_rows is not None:
        pos = pl.DataFrame({"row": np.asarray(a_rows, np.int32), "a": np.arange(len(a_rows), dtype=np.int32)})
        ka = ka.join(pos, on="row").drop("row")
    else:
        ka = ka.rename({"row": "a"})
    kb = KB.filter(pl.col("cnt") <= KCAP).rename({"row": "b", "cnt": "cb"})
    J = ka.rename({"cnt": "ca"}).join(kb.drop("kt"), on="kh")
    if J.height == 0:
        return None
    agg = []
    for t, nm in enumerate(KEY_TYPES):
        m = pl.col("kt") == t
        agg += [m.any().cast(pl.Int8).alias(f"k_{nm}"),
                pl.col("ca").filter(m).min().alias(f"k_{nm}_ca"),
                pl.col("cb").filter(m).min().alias(f"k_{nm}_cb")]
    agg += [(pl.col("kt") == 4).sum().cast(pl.Int16).alias("k_addr_tok_n"), pl.len().cast(pl.Int16).alias("k_n")]
    return J.group_by(["a", "b"]).agg(agg).to_pandas()

KEY_FEATS = [f"k_{nm}{s}" for nm in KEY_TYPES for s in ("", "_ca", "_cb")] + ["k_addr_tok_n", "k_n"]

def merge_key_pairs(P, KP, key_only_types=("name", "nameset", "name_num", "addr_tok", "tok2_num")):
    """Add key features to the TF-IDF shortlist P(a, b, sim_name, sim_addr), and add key-only pairs
    (pairs the TF-IDF shortlist missed) when a name-based key matched."""
    if KP is None or len(KP) == 0:
        for c in KEY_FEATS:
            P[c] = np.nan if c.endswith(("_ca", "_cb")) else 0
        P["from_tfidf"] = np.int8(1)
        return P
    KP = KP.copy()
    rare = [(KP[f"k_{t}"].to_numpy() == 1) & (KP[f"k_{t}_ca"].fillna(1e9).to_numpy() <= 1)
            & (KP[f"k_{t}_cb"].fillna(1e9).to_numpy() <= 3) for t in key_only_types]
    extra = KP[np.logical_or.reduce(rare)]                   # only rare keys may add pairs the shortlist missed
    P = P.merge(KP, on=["a", "b"], how="left")
    P["from_tfidf"] = np.int8(1)
    new = extra.merge(P[["a", "b"]], on=["a", "b"], how="left", indicator=True)
    new = new[new._merge == "left_only"].drop(columns="_merge")
    new["sim_name"] = np.nan; new["sim_addr"] = np.nan; new["from_tfidf"] = np.int8(0)
    P = pd.concat([P, new[P.columns]], ignore_index=True)
    for t in KEY_TYPES:
        P[f"k_{t}"] = P[f"k_{t}"].fillna(0).astype(np.int8)
    P["k_addr_tok_n"] = P["k_addr_tok_n"].fillna(0).astype(np.int16)
    P["k_n"] = P["k_n"].fillna(0).astype(np.int16)
    P["a"] = P.a.astype(np.int64); P["b"] = P.b.astype(np.int64)
    return P

# ---------------------------------------------------------------------
# Exactness checks. The data contains planted look-alikes: same name and street, but a different
# house number (712 vs 719) or a different legal form (LLP vs Private Limited). v1 threw legal words away.
# ---------------------------------------------------------------------
LEGAL_MAP = {"prwt": "pvt", "pwt": "pvt", "prvt": "pvt", "lmtd": "ltd", "ltd": "ltd", "lmt": "ltd", "llp": "llp",
             "llc": "llc", "inc": "inc", "ncrprtd": "inc", "crp": "corp", "crprtn": "corp", "cmpny": "co", "c": "co",
             "plc": "plc", "pc": "pc", "pllc": "pllc", "lp": "lp", "srl": "sarl", "eurl": "eurl", "ss": "sas",
             "ssu": "sas", "sa": "sa", "gmbh": "gmbh", "snc": "snc", "sc": "sci", "sk": "sci", "dba": "dba"}

def _legal_rows(rows):
    out = []
    for nb, an in rows:
        lg = set(LEGAL_MAP.get(skel_tok(t)) for t in nb.split()) - {None}
        if "pvt" in lg: lg.discard("ltd")                  # "Private Limited" is one legal form
        nums = [x.lstrip("0") or "0" for x in re.findall(r"\d+", an)]
        out.append(("|".join(sorted(lg)), " ".join(nums)))
    return out

def add_extra(D):
    """Adds per-record columns: legal (legal form, e.g. 'pvt', 'llp|co') and nums (numbers in the address)."""
    rows = list(zip(D.name_b.tolist(), D.addr_n.tolist())); step = 100_000
    with Pool(N_CPU) as pool:
        res = [r for ch in pool.map(_legal_rows, [rows[i:i + step] for i in range(0, len(rows), step)]) for r in ch]
    D["legal"] = [r[0] for r in res]; D["nums"] = [r[1] for r in res]
    return D

def _near(x, y):
    """Two different numbers that look like a planted change: close in value, or same length and one digit apart."""
    if x == y: return False
    if abs(int(x[:9]) - int(y[:9])) <= 20: return True
    return len(x) == len(y) and sum(p != q for p, q in zip(x, y)) == 1

EXACT_FEATS = ["legal_rel", "legal_s1", "legal_cand", "num_eq", "num_cand_extra", "num_s1_extra", "num_first_eq",
               "num_near", "x_noextra", "x_legal_same"]

def _exact_rows(args):
    out = np.zeros((len(args[0]), 8), np.int16)
    for i, (x, y, p, q) in enumerate(zip(*args)):
        lr = -1 if (not x or not y) else int(x == y)
        pl_, ql = p.split(), q.split(); ps, qs = set(pl_), set(ql)
        ea, eb = ps - qs, qs - ps
        first = -1 if (not pl_ or not ql) else int(pl_[0] == ql[0])
        near = int(any(_near(u, v) for u in ea for v in eb)) if ea and eb else 0
        out[i] = (lr, int(bool(x)), int(bool(y)), int(ps == qs), len(eb), len(ea), first, near)
    return out

def exact_features(P, A, B):
    a, b = P.a.to_numpy(), P.b.to_numpy()
    cols = [A.legal.to_numpy()[a], B.legal.to_numpy()[b], A.nums.to_numpy()[a], B.nums.to_numpy()[b]]
    step = 200_000
    with Pool(N_CPU) as pool:
        res = pool.map(_exact_rows, [[c[i:i + step].tolist() for c in cols] for i in range(0, len(P), step)])
    F = pd.DataFrame(np.concatenate(res) if res else np.zeros((0, 8), np.int16), columns=EXACT_FEATS[:8])
    F["x_noextra"] = ((F.num_cand_extra.to_numpy() == 0) & (pd.Series(cols[3]).str.len().to_numpy() > 0)).astype(np.int8)
    F["x_legal_same"] = (F.legal_rel.to_numpy() == 1).astype(np.int8)
    return F

# =====================================================================
# v14: one pair table with the features of v1 + v6 + v7 + v8 together
# =====================================================================
def add_name_freq(D, ref=None):
    """v6: how many records per 100k of the same country share this cleaned name (ref = table to count in)."""
    ref = D if ref is None else ref
    cnt = ref.groupby(["country", "name_skc"]).size()
    tot = ref.groupby("country").size()
    key = pd.MultiIndex.from_arrays([D.country, D.name_skc])
    c = cnt.reindex(key).fillna(0).to_numpy()
    D["name_freq"] = (c / tot.reindex(D.country).fillna(1).to_numpy() * 1e5).astype(np.float32)
    return D

def prep_all(D):
    """All cleaned columns every version needs."""
    return add_name_freq(add_extra(prepare_extra(prepare(D))))

def pair_table(Ac, a_rows, KA, Bc, bpos, KB, is_s3, idx=None, R=None):
    """Shortlist (TF-IDF + v7 keys) and every feature for S1 rows Ac against one country pool Bc.
    a_rows: rows of Ac inside the table KA was built on; bpos: rows of Bc inside the table KB was built on.
    idx / R: Index and Rarity of Bc, when already built (test time, many batches)."""
    P = (idx or Index(Bc)).query(Ac)
    KP = key_pairs(KA, KB, a_rows)
    if KP is not None:
        inv = np.full(int(bpos.max()) + 1 if len(bpos) else 0, -1, np.int64); inv[bpos] = np.arange(len(bpos))
        bb = KP.b.to_numpy()
        ok = bb < len(inv)
        KP = KP[ok]; KP["b"] = inv[KP.b.to_numpy()]; KP = KP[KP.b.to_numpy() >= 0]
    P = merge_key_pairs(P, KP)
    F = features_v8(P, Ac, Bc, is_s3, R or Rarity(Bc))
    a, b = P.a.to_numpy(), P.b.to_numpy()
    for c in KEY_FEATS + ["from_tfidf"]:
        F[c] = P[c].to_numpy()
    E = exact_features(P, Ac, Bc)
    for c in EXACT_FEATS:
        F[c] = E[c].to_numpy()
    F["s1_name_freq"] = Ac.name_freq.to_numpy()[a]
    F["cand_name_freq"] = Bc.name_freq.to_numpy()[b]
    F["n_name_close"] = pd.Series(F.nskc_ratio.to_numpy() >= 90).groupby(a).transform("sum").to_numpy(np.float32)
    F["cand_noaddr"] = (Bc.addr_n.to_numpy()[b] == "").astype(np.int8)
    return P, F.astype(np.float32)

# ---------------------------------------------------------------------
# v2 stage 2: context of the business's other strong candidates (copies of one business look alike)
# ---------------------------------------------------------------------
S2_EXTRA = ["p1", "p1_rank", "p1_max", "p1_2nd", "p1_sum", "p1_rel", "n_hi", "n_kept",
            "sup_max", "sup_sum", "sup_cnt", "sup_name_max", "sup_addr_max", "sup_other_src"]

def stage2_features(key, p1, cname, caddr, is_s3, K=8):
    d = pd.DataFrame({"key": key, "p1": p1.astype(np.float32), "row": np.arange(len(key)), "s3": is_s3})
    d = d.sort_values(["key", "p1"], ascending=[True, False], kind="stable")
    g = d.groupby("key", sort=False)
    d["p1_rank"] = (g.cumcount() + 1).astype(np.float32)
    d["p1_max"] = g.p1.transform("max")
    d["p1_2nd"] = d.key.map(d[d.p1_rank == 2].set_index("key").p1).fillna(0).to_numpy(np.float32)
    d["p1_sum"] = g.p1.transform("sum")
    d["p1_rel"] = d.p1 / d.p1_max.clip(lower=1e-6)
    d["n_hi"] = (d.p1 > 0.5).groupby(d.key).transform("sum").astype(np.float32)
    d["n_kept"] = g.p1.transform("size").astype(np.float32)
    top = d[d.p1_rank <= K][["key", "row", "p1", "s3"]]
    j = top.merge(top, on="key", suffixes=("", "_o"))
    j = j[j.row != j.row_o]
    r, ro = j.row.to_numpy(), j.row_o.to_numpy()
    sn = cpdist(cname[r].tolist(), cname[ro].tolist(), scorer=fuzz.token_set_ratio, workers=N_CPU).astype(np.float32) / 100
    sa = cpdist(caddr[r].tolist(), caddr[ro].tolist(), scorer=fuzz.token_set_ratio, workers=N_CPU).astype(np.float32) / 100
    po = j.p1_o.to_numpy()
    s = (sn + sa) / 2
    j = pd.DataFrame({"row": r, "w": po * s, "strong": ((s > 0.85) & (po > 0.5)).astype(np.int8),
                      "wn": po * sn, "wa": po * sa, "wx": np.where(j.s3.to_numpy() != j.s3_o.to_numpy(), po * s, 0)})
    agg = j.groupby("row").agg(sup_max=("w", "max"), sup_sum=("w", "sum"), sup_cnt=("strong", "sum"),
                               sup_name_max=("wn", "max"), sup_addr_max=("wa", "max"), sup_other_src=("wx", "max"))
    d = d.set_index("row").sort_index().join(agg)
    return d[S2_EXTRA].fillna(0).astype(np.float32).reset_index(drop=True)

def stage2_in_chunks(key, *cols, chunk=200_000, K=8):
    order = np.argsort(key, kind="stable"); ks = key[order]
    uk = np.unique(ks); out = []
    for i in range(0, len(uk), chunk):
        lo, hi = np.searchsorted(ks, uk[i]), np.searchsorted(ks, uk[min(i + chunk, len(uk)) - 1], side="right")
        sel = order[lo:hi]
        f = stage2_features(key[sel], *[c[sel] for c in cols], K=K); f.index = sel; out.append(f)
    return pd.concat(out).sort_index().reset_index(drop=True)


In [ ]:
# =====================================================================
# v14 step 1: pair table + all features for the 150k training businesses (same sample as v1/v8),
# saved to disk block by block. Validation = the same 20% held-out businesses (rng 42).
# =====================================================================
ROOT = os.environ.get("ROOT", "/kaggle/input"); WORK = os.environ.get("WORK", "/kaggle/working")
N_TRAIN_S1 = int(os.environ.get("N_TRAIN_S1", 150_000)); SMOKE = 0
BLK = f"{WORK}/v14_blocks"; os.makedirs(BLK, exist_ok=True)
T0 = time.time()
if not os.path.exists(f"{BLK}/done"):
    log("loading train Source 1 + ground truth")
    s1 = load(ROOT, "train_source1"); gt = load(ROOT, "train_ground_truth")
    truth = {r.source1_entity_id: set(x for x in r.matched_entity_ids.split(",") if x) for r in gt.itertuples()}
    del gt; gc.collect()
    samp = s1.sample(4000 if SMOKE else N_TRAIN_S1, random_state=0).index.to_numpy()   # same sample as v1
    S1 = prep_all(s1); del s1; free_mem(); log("prepared all train S1", S1.shape)
    KA = make_keys(S1).filter(pl.col("row").is_in(samp))   # key counts over ALL S1, then keep the sample's keys
    A = S1.iloc[samp].reset_index(drop=True); del S1; free_mem()
    json.dump({s: sorted(truth[s]) for s in A.entity_id}, open(f"{WORK}/v14_truth.json", "w"))
    for src, is_s3 in (("train_source2", 0), ("train_source3", 1)):
        if os.path.exists(f"{BLK}/{src}.done"): continue
        log("loading", src); B = load(ROOT, src)
        if SMOKE:
            want = set().union(*(truth[s] for s in A.entity_id))
            B = pd.concat([B[B.entity_id.isin(want)], B.sample(200_000, random_state=0)]).drop_duplicates("entity_id")
            B = B.reset_index(drop=True)
        B = prep_all(B); log("prepared", src, B.shape)
        KB = make_keys(B); log(src, "keys", KB.height)
        for country in A.country.unique():
            sel = (A.country == country).to_numpy()
            Ac = A[sel].reset_index(drop=True); bpos = np.flatnonzero((B.country == country).to_numpy())
            if len(Ac) == 0 or len(bpos) == 0: continue
            Bc = B.iloc[bpos].reset_index(drop=True)
            P, F = pair_table(Ac, samp[sel], KA, Bc, bpos, KB, is_s3)
            b = P.b.to_numpy()
            m = pd.DataFrame({"s1_id": Ac.entity_id.to_numpy()[P.a.to_numpy()], "cand_id": Bc.entity_id.to_numpy()[b],
                              "cname": Bc.name_sk.to_numpy()[b], "caddr": Bc.addr_sk.to_numpy()[b],
                              "country": country, "s3": np.int8(is_s3)})
            m["label"] = np.fromiter((c in truth.get(s, ()) for s, c in zip(m.s1_id, m.cand_id)), bool, len(m))
            F.to_parquet(f"{BLK}/X_{src}_{country}.parquet"); m.to_parquet(f"{BLK}/M_{src}_{country}.parquet")
            log(src, country, "pairs", len(P), "(keys only", int((F.from_tfidf == 0).sum()), ") true", int(m.label.sum()),
                "| features", F.shape[1], "| min", round((time.time() - T0) / 60, 1))
            del P, F, m, Ac, Bc; free_mem()
        open(f"{BLK}/{src}.done", "w").close()
        del B, KB; free_mem()
    open(f"{BLK}/done", "w").close()
log("pair table ready | minutes", round((time.time() - T0) / 60, 1))


In [ ]:
# =====================================================================
# v14 step 2: train on the pair table from step 1.
# LightGBM and XGBoost on all 105 features (v1 + v6 + v7 + v8 together); the notebook keeps whichever of
# LightGBM / XGBoost / their average scores best on the same 30k held-out businesses as v1 (0.9529).
# =====================================================================
LR = float(os.environ.get("LR", 0.08))
T0 = time.time()
truth = {s: set(v) for s, v in json.load(open(f"{WORK}/v14_truth.json")).items()}
ids = np.array(list(truth))
val_ids = set(np.random.default_rng(42).choice(ids, int(0.2 * len(ids)), replace=False))   # same as v1 / v8
vl = sorted(val_ids); vt = {s: truth[s] for s in vl}
blocks = sorted(glob.glob(f"{BLK}/X_*.parquet"))
M = pd.concat([pd.read_parquet(f.replace("/X_", "/M_"), columns=["s1_id", "cand_id", "label"]) for f in blocks], ignore_index=True)
FEATS = list(pd.read_parquet(blocks[0]).columns)
iv0 = M.s1_id.isin(val_ids).to_numpy()
perm = np.r_[np.flatnonzero(~iv0), np.flatnonzero(iv0)]; n_tr = int((~iv0).sum())   # training rows first
dest = np.empty(len(perm), np.int64); dest[perm] = np.arange(len(perm))
XP = f"{WORK}/v14_X.npy"
if not os.path.exists(XP):          # all features in one file on disk, read on demand (keeps memory free)
    Xw = np.lib.format.open_memmap(XP + ".tmp", "w+", np.float32, (len(M), len(FEATS))); o = 0
    for f in blocks:
        F = pd.read_parquet(f)[FEATS].to_numpy(np.float32); Xw[dest[o:o + len(F)]] = F; o += len(F); del F; gc.collect()
    Xw.flush(); del Xw; os.rename(XP + ".tmp", XP)
Xn = np.load(XP, mmap_mode="r")
M = M.iloc[perm].reset_index(drop=True); del perm, dest; gc.collect()
y = M.label.to_numpy().astype(np.int8)
Mv = M.iloc[n_tr:][["s1_id", "cand_id"]].reset_index(drop=True)
log("pairs", len(M), "| features", len(FEATS), "| shortlist recall", round(y.sum() / sum(len(v) for v in truth.values()), 4))
log("baseline all-empty", round(macro_f05({}, vt, vl), 4),
    "| ceiling (perfect model on this shortlist)", round(macro_f05(Mv[y[n_tr:] == 1].groupby("s1_id").cand_id.apply(set).to_dict(), vt, vl), 4))
GRID = np.round(np.arange(0.30, 0.901, 0.025), 3)

def best_cut(p, label):
    pv = Mv.assign(p=p)
    t, s = max(((t, macro_f05(decide(pv, t), vt, vl)) for t in GRID), key=lambda r: r[1])
    log(f"BEST val F0.5 [{label}] = {s:.4f} at cut-off {t}"); return float(t), float(s)

Xva = np.asarray(Xn[n_tr:]); PV, RES = {}, {}
t1 = time.time(); log("training LightGBM (all features)")
m = lgb.LGBMClassifier(n_estimators=4000 if not SMOKE else 300, learning_rate=LR, num_leaves=127, min_child_samples=50,
                       subsample=0.7, subsample_freq=1, colsample_bytree=0.5, reg_lambda=5.0, n_jobs=N_CPU, verbose=-1)
m.fit(Xn[:n_tr], y[:n_tr], eval_set=[(Xva, y[n_tr:])], callbacks=[lgb.early_stopping(100, verbose=False)])
PV["lgb"] = m.predict_proba(Xva)[:, 1].astype(np.float32); RES["lgb"] = best_cut(PV["lgb"], "LightGBM")
m.booster_.save_model(f"{WORK}/v14_lgb.txt"); log("trees", m.best_iteration_, "| minutes", round((time.time() - t1) / 60, 1))
imp = pd.Series(m.booster_.feature_importance("gain"), FEATS); imp = (imp / imp.sum()).sort_values(ascending=False)
log("top features:\n" + imp.head(15).round(4).to_string()); del m; gc.collect()
t1 = time.time(); log("training XGBoost (all features)")
dtr = xgb.QuantileDMatrix(Xn[:n_tr], y[:n_tr], max_bin=256, nthread=N_CPU); dva = xgb.DMatrix(Xva, y[n_tr:], nthread=N_CPU)
bx = xgb.train(dict(eta=0.08, max_depth=9, min_child_weight=5, subsample=0.8, colsample_bytree=0.7, tree_method="hist",
                    max_bin=256, eval_metric="logloss", objective="binary:logistic", seed=2, nthread=N_CPU),
               dtr, 4000 if not SMOKE else 300, evals=[(dva, "val")], early_stopping_rounds=100, verbose_eval=False)
bx = bx[: bx.best_iteration + 1]; bx.save_model(f"{WORK}/v14_xgb.json")
PV["xgb"] = bx.predict(dva).astype(np.float32); RES["xgb"] = best_cut(PV["xgb"], "XGBoost")
log("trees", bx.num_boosted_rounds(), "| minutes", round((time.time() - t1) / 60, 1)); del dtr, dva, bx; gc.collect()
RES["avg"] = best_cut((PV["lgb"] + PV["xgb"]) / 2, "average of LightGBM + XGBoost")
choice = max(RES, key=lambda k: RES[k][1])
USE = {"lgb": ["lgb"], "xgb": ["xgb"], "avg": ["lgb", "xgb"]}[choice]
log(f"FINAL choice: {choice}, cut-off {RES[choice][0]}, val F0.5 {RES[choice][1]:.4f} (v1 was 0.9529, v8 0.9673)")
json.dump({"features": FEATS, "models": USE, "threshold": RES[choice][0], "results": RES},
          open(f"{WORK}/v14_meta.json", "w"), indent=1)
del Xn, Xva, M, y, PV; gc.collect()
log("training finished | minutes", round((time.time() - T0) / 60, 1))


In [ ]:
# =====================================================================
# v14 step 3: every test business, in batches saved to disk (safe to resume), then the output files.
# =====================================================================
BATCH = int(os.environ.get("BATCH", 100_000))
UNSEEN_MARGIN = 0.10          # France (no training answers): cut-off raised by this much, to stay on the safe side
CAND_K, CAND_P = 10, 0.001    # candidate file: per business the 10 best-scored candidates with p >= 0.001 (+ all matches)
T0 = time.time()
meta = json.load(open(f"{WORK}/v14_meta.json")); FEATS = meta["features"]
MODELS = []
if "lgb" in meta["models"]: MODELS.append(("lgb", lgb.Booster(model_file=f"{WORK}/v14_lgb.txt")))
if "xgb" in meta["models"]:
    bx = xgb.Booster(); bx.load_model(f"{WORK}/v14_xgb.json"); bx.set_param({"nthread": N_CPU}); MODELS.append(("xgb", bx))
log("models", [n for n, _ in MODELS], "| cut-off", meta["threshold"], "| unseen-country cut-off", meta["threshold"] + UNSEEN_MARGIN)

def score(F):
    out = []
    for n, m in MODELS:
        out.append(m.predict(F, num_threads=N_CPU) if n == "lgb" else m.predict(xgb.DMatrix(F, nthread=N_CPU)))
    return np.mean(out, axis=0).astype(np.float32)

PARTS = f"{WORK}/parts_v14"; os.makedirs(PARTS, exist_ok=True)
log("loading test Source 1"); A = load(ROOT, "test_source1")
s1_ids = A.entity_id.to_numpy()
if SMOKE: A = A.groupby("country").head(1500)
A = prep_all(A); log("prepared", A.shape); print(A.country.value_counts().to_string())
A_rowid = pd.Series(np.arange(len(s1_ids)), index=s1_ids).reindex(A.entity_id).to_numpy(np.int32)
KA = None
for src, is_s3 in (("test_source2", 0), ("test_source3", 1)):
    todo = [(c, i) for c in A.country.unique() for i in range(0, int((A.country == c).sum()), BATCH)
            if not os.path.exists(f"{PARTS}/{src}_{c}_{i}.parquet")]
    if not todo: log(src, "already done"); continue
    if KA is None: KA = make_keys(A); log("S1 keys", KA.height)
    log("loading", src); B = load(ROOT, src)
    if SMOKE: B = B.sample(300_000, random_state=0).reset_index(drop=True)
    B = prep_all(B); log("prepared", src, B.shape)
    KB = make_keys(B); log(src, "keys", KB.height)
    for country in A.country.unique():                 # any country works, France included
        if not any(c == country for c, _ in todo): continue
        a_rows = np.flatnonzero(A.country.to_numpy() == country)
        bpos = np.flatnonzero((B.country == country).to_numpy())
        if len(bpos) == 0:
            for i in range(0, len(a_rows), BATCH):
                pd.DataFrame({"s1_row": np.zeros(0, np.int32), "cand_id": [], "p": np.zeros(0, np.float32)}).to_parquet(f"{PARTS}/{src}_{country}_{i}.parquet")
            continue
        Bc = B.iloc[bpos].reset_index(drop=True); idx = Index(Bc); R = Rarity(Bc); bids = Bc.entity_id.to_numpy()
        nb = -(-len(a_rows) // BATCH)
        for i in range(0, len(a_rows), BATCH):
            out = f"{PARTS}/{src}_{country}_{i}.parquet"
            if os.path.exists(out): continue
            rows = a_rows[i:i + BATCH]; Ac = A.iloc[rows].reset_index(drop=True)
            P, F = pair_table(Ac, rows, KA, Bc, bpos, KB, is_s3, idx=idx, R=R)
            p = score(F[FEATS].to_numpy(np.float32)) if len(P) else np.zeros(0, np.float32); del F
            pd.DataFrame({"s1_row": A_rowid[rows[P.a.to_numpy()]], "cand_id": bids[P.b.to_numpy()], "p": p}).to_parquet(out)
            log(src, country, f"batch {i // BATCH + 1}/{nb}", "pairs", len(P), "| minutes", round((time.time() - T0) / 60, 1))
            del P, p, Ac; free_mem()
        del idx, R, Bc; free_mem()
    del B, KB; free_mem()
del KA; gc.collect()

# ---------------- combine (low memory) ----------------
country_of = pl.DataFrame({"s1_row": A_rowid, "country": A.country.to_numpy()})
del A; free_mem()
thr = meta["threshold"]
cut = pl.when(pl.col("country").is_in(["US", "India"])).then(pl.lit(thr)).otherwise(pl.lit(min(thr + UNSEEN_MARGIN, 0.97)))
sc = pl.scan_parquet(f"{PARTS}/*.parquet").filter(pl.col("p") >= min(CAND_P, 0.05)).collect()
sc = sc.join(country_of, on="s1_row", how="left").with_columns(cut.alias("cut"))
# 1) matches: p >= cut-off; each Source 2/3 record goes to its best business only
best = (sc.filter(pl.col("p") >= pl.col("cut")).sort("p", descending=True)
          .unique(subset="cand_id", keep="first", maintain_order=True))
# 2) candidates: top CAND_K per business (p >= CAND_P) plus every match
top = sc.filter(pl.col("p") >= CAND_P).sort("p", descending=True).group_by("s1_row", maintain_order=True).head(CAND_K)
cand = pl.concat([top.select("s1_row", "cand_id"), best.select("s1_row", "cand_id")]).unique()
# 3) scores for the v14 merge notebook (and for later blending)
ids_t = pl.DataFrame({"s1_row": np.arange(len(s1_ids), dtype=np.int32), "s1_id": s1_ids})
sc.filter(pl.col("p") >= 0.05).join(ids_t, on="s1_row", how="left") \
  .select("s1_id", "cand_id", "p", "country", "cut").write_parquet(f"{WORK}/test_scores.parquet"); del ids_t
os.makedirs(f"{WORK}/output", exist_ok=True)
def write(path, table, col):
    g = table.group_by("s1_row").agg(pl.col("cand_id").sort().str.join(","))
    line = np.full(len(s1_ids), "", dtype=object)
    for r, c in zip(g["s1_row"].to_list(), g["cand_id"].to_list()): line[r] = c
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(f"source1_entity_id\t{col}\n")
        for s, m in zip(s1_ids, line): fh.write(s + "\t" + m + "\n")
    return int((line != "").sum())
n = write(f"{WORK}/output/matching_results.tsv", best, "matched_entity_ids")
write(f"{WORK}/output/candidate_pairs.tsv", cand, "candidate_entity_ids")
log("done:", len(s1_ids), "rows,", n, "with matches,", len(s1_ids) - n, "empty; avg matches", round(best.height / len(s1_ids), 2),
    "| CANDIDATE SET: avg", round(cand.height / len(s1_ids), 2), "per business | predict minutes", round((time.time() - T0) / 60, 1))
print(best.group_by("country").agg(pl.len().alias("matches")).sort("country"))


In [ ]:
validator = glob.glob(f'{ROOT}/**/validate_submission.py', recursive=True)[0]
test_dir = os.path.dirname(find_file(ROOT, 'test_source1')[0])
!python3 {validator} --matching {WORK}/output/matching_results.tsv --candidate {WORK}/output/candidate_pairs.tsv --test-dir {test_dir}

**Done.** Output tab → `output/`: download **`matching_results.tsv`** (to upload) and **`candidate_pairs.tsv`** (for the final zip). `test_scores.parquet` is for the v14 merge notebook. You can delete `parts_v14/`, `v14_blocks/` and `v14_X.npy`.